# Ψ-NeuroFractal : Detection precoce Alzheimer/Parkinson
## Entrainement du modele ΨTA vs CNN Baseline

In [ ]:
!git clone https://github.com/hassanegbane190-max/psi-neurofractal.git
%cd psi-neurofractal
!pip install torch numpy scipy scikit-learn matplotlib

## 1. Entrainement du modele ΨTA

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from scipy.signal import hilbert

np.random.seed(42)
torch.manual_seed(42)

# ============================================================
# NEURONE ΨTA (Asymmetric Psi-Transcendence)
# ============================================================
class AsymmetricPsiTNeuron(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weights = nn.Parameter(torch.randn(out_features, in_features, dtype=torch.complex64) * 0.8)
        self.bias = nn.Parameter(torch.zeros(out_features, dtype=torch.complex64))
        self.alpha = nn.Parameter(torch.ones(out_features) * 1.5)
        self.beta = nn.Parameter(torch.zeros(out_features))
        self.gamma = nn.Parameter(torch.ones(out_features) * 2.0)

    def forward(self, x):
        z = torch.matmul(x, self.weights.t()) + self.bias
        R = torch.abs(z)
        theta = torch.angle(z)
        amplitude = torch.tanh(R)
        angle_asym = torch.relu(theta + self.beta)
        phi_att = np.pi * self.gamma * torch.tanh(self.alpha * R) * torch.sin(angle_asym)
        return amplitude * torch.complex(torch.cos(phi_att), torch.sin(phi_att))

# ============================================================
# DATASET EEG SIMULE
# ============================================================
class EEGAlzheimerDataset(Dataset):
    def __init__(self, n_channels=64, n_times=256, n_samples_per_class=500):
        self.samples = []
        self.labels = []
        fs = 256.0
        t = np.arange(n_times) / fs

        for class_label in range(2):
            for _ in range(n_samples_per_class):
                eeg = np.zeros((n_channels, n_times))
                for ch in range(n_channels):
                    if class_label == 0:
                        alpha = 10.0 * np.sin(2*np.pi*10*t + np.random.uniform(0, 2*np.pi))
                        beta = 3.0 * np.sin(2*np.pi*20*t + np.random.uniform(0, 2*np.pi))
                        theta = 4.0 * np.sin(2*np.pi*6*t + np.random.uniform(0, 2*np.pi))
                        eeg[ch] = alpha + beta + theta + 0.5*np.random.randn(len(t))
                    else:
                        alpha = 2.0 * np.sin(2*np.pi*10*t + np.random.uniform(0, 2*np.pi))
                        theta = 8.0 * np.sin(2*np.pi*5*t + np.random.uniform(0, 2*np.pi))
                        delta = 6.0 * np.sin(2*np.pi*2*t + np.random.uniform(0, 2*np.pi))
                        chaos = sum((1.0/k)*np.sin(2*np.pi*k*3.7*t + np.random.uniform(0, 2*np.pi)) for k in range(1, 6))
                        eeg[ch] = alpha + theta + delta + chaos + 1.5*np.random.randn(len(t))

                complex_signals = np.zeros((n_channels, n_times), dtype=np.complex64)
                for ch in range(n_channels):
                    complex_signals[ch] = hilbert(eeg[ch]).astype(np.complex64)

                self.samples.append(torch.tensor(complex_signals, dtype=torch.complex64))
                self.labels.append(class_label)

        self.samples = torch.stack(self.samples)
        self.labels = torch.tensor(self.labels, dtype=torch.long)

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.samples[idx], self.labels[idx]

# ============================================================
# MODELES
# ============================================================
class NeuroFractalNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.input_proj = nn.Linear(256, 256, dtype=torch.complex64)
        self.psi1 = AsymmetricPsiTNeuron(64, 128)
        self.psi2 = AsymmetricPsiTNeuron(128, 64)
        self.norm1 = nn.LayerNorm([256])
        self.norm2 = nn.LayerNorm([128])
        self.classifier = nn.Sequential(
            nn.Linear(128, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(16, 2)
        )

    def forward(self, x):
        if x.dim() == 3: x = x.squeeze(1)
        x = self.input_proj(x)
        h = self.psi1(x)
        h_real = torch.cat([torch.real(h), torch.imag(h)], dim=1)
        h_real = self.norm1(h_real)
        h = self.psi2(h)
        h_real = torch.cat([torch.real(h), torch.imag(h)], dim=1)
        h_real = self.norm2(h_real)
        return self.classifier(h_real)

class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3))
        self.classifier = nn.Sequential(nn.Linear(64*64, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 32), nn.ReLU(), nn.Linear(32, 2))

    def forward(self, x):
        if x.dim() == 3: x = x.squeeze(1)
        if x.is_complex(): x = torch.cat([torch.real(x), torch.imag(x)], dim=1)
        x = x.float()
        features = [self.features(x[:, ch, :]) for ch in range(x.shape[1])]
        return self.classifier(torch.cat(features, dim=1))

print('Definitions chargees avec succes !')

## 2. Generation des donnees EEG

In [ ]:
print('Generation des donnees EEG...')
dataset = EEGAlzheimerDataset()
print(f'Dataset : {len(dataset)} echantillons')
print(f'Sample shape : {dataset.samples.shape}')
print(f'Labels : {torch.bincount(dataset.labels)}')

n_train = 800
n_test = 200
train_loader = DataLoader(torch.utils.data.Subset(dataset, range(n_train)), batch_size=32, shuffle=True)
test_loader = DataLoader(torch.utils.data.Subset(dataset, range(n_train, n_train+n_test)), batch_size=32, shuffle=False)
print(f'Train : {n_train} | Test : {n_test}')

## 3. Entrainement ΨTA

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

model_psita = NeuroFractalNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_psita.parameters(), lr=0.001, weight_decay=1e-5)

print(f'Parametres ΨTA : {sum(p.numel() for p in model_psita.parameters()):,}')
print('\n' + '='*65)
print(f'{"Epoch":>6} | {"Loss Train":>10} | {"Acc Train":>10} | {"Loss Test":>10} | {"Acc Test":>10}')
print('='*65)

history_psita = {'train_loss':[], 'train_acc':[], 'test_loss':[], 'test_acc':[]}
best_acc_psita = 0
start = time.time()

for epoch in range(1, 101):
    model_psita.train()
    total_loss = correct = total = 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        out = model_psita(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()*bx.size(0)
        correct += (out.argmax(1)==by).sum().item()
        total += by.size(0)
    train_loss = total_loss/total
    train_acc = correct/total*100

    model_psita.eval()
    total_loss = correct = total = 0
    with torch.no_grad():
        for bx, by in test_loader:
            bx, by = bx.to(device), by.to(device)
            out = model_psita(bx)
            loss = criterion(out, by)
            total_loss += loss.item()*bx.size(0)
            correct += (out.argmax(1)==by).sum().item()
            total += by.size(0)
    test_loss = total_loss/total
    test_acc = correct/total*100

    history_psita['train_loss'].append(train_loss)
    history_psita['train_acc'].append(train_acc)
    history_psita['test_loss'].append(test_loss)
    history_psita['test_acc'].append(test_acc)

    if test_acc > best_acc_psita:
        best_acc_psita = test_acc
        torch.save(model_psita.state_dict(), 'psita_best.pt')

    if epoch % 10 == 0 or epoch == 1:
        print(f'{epoch:6d} | {train_loss:10.4f} | {train_acc:9.2f}% | {test_loss:10.4f} | {test_acc:9.2f}%')

print('='*65)
print(f'\nΨTA termine en {time.time()-start:.1f}s | Meilleure accuracy : {best_acc_psita:.2f}%')

## 4. Entrainement CNN Baseline

In [ ]:
model_cnn = BaselineCNN().to(device)
optimizer_cnn = torch.optim.Adam(model_cnn.parameters(), lr=0.001, weight_decay=1e-5)

print(f'Parametres CNN : {sum(p.numel() for p in model_cnn.parameters()):,}')
print('\n' + '='*65)
print(f'{"Epoch":>6} | {"Loss Train":>10} | {"Acc Train":>10} | {"Loss Test":>10} | {"Acc Test":>10}')
print('='*65)

history_cnn = {'train_loss':[], 'train_acc':[], 'test_loss':[], 'test_acc':[]}
best_acc_cnn = 0
start = time.time()

for epoch in range(1, 101):
    model_cnn.train()
    total_loss = correct = total = 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer_cnn.zero_grad()
        out = model_cnn(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer_cnn.step()
        total_loss += loss.item()*bx.size(0)
        correct += (out.argmax(1)==by).sum().item()
        total += by.size(0)
    train_loss = total_loss/total
    train_acc = correct/total*100

    model_cnn.eval()
    total_loss = correct = total = 0
    with torch.no_grad():
        for bx, by in test_loader:
            bx, by = bx.to(device), by.to(device)
            out = model_cnn(bx)
            loss = criterion(out, by)
            total_loss += loss.item()*bx.size(0)
            correct += (out.argmax(1)==by).sum().item()
            total += by.size(0)
    test_loss = total_loss/total
    test_acc = correct/total*100

    history_cnn['train_loss'].append(train_loss)
    history_cnn['train_acc'].append(train_acc)
    history_cnn['test_loss'].append(test_loss)
    history_cnn['test_acc'].append(test_acc)

    if test_acc > best_acc_cnn:
        best_acc_cnn = test_acc

    if epoch % 10 == 0 or epoch == 1:
        print(f'{epoch:6d} | {train_loss:10.4f} | {train_acc:9.2f}% | {test_loss:10.4f} | {test_acc:9.2f}%')

print('='*65)
print(f'\nCNN termine en {time.time()-start:.1f}s | Meilleure accuracy : {best_acc_cnn:.2f}%')

## 5. Comparaison Finale + Graphiques

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(history_psita['train_loss'], label='Train', color='blue')
axes[0,0].plot(history_psita['test_loss'], label='Test', color='red')
axes[0,0].set_title('Loss - ΨTA', fontweight='bold')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(history_psita['train_acc'], label='Train', color='blue')
axes[0,1].plot(history_psita['test_acc'], label='Test', color='red')
axes[0,1].set_title('Accuracy - ΨTA', fontweight='bold')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(history_cnn['train_loss'], label='Train', color='blue')
axes[1,0].plot(history_cnn['test_loss'], label='Test', color='red')
axes[1,0].set_title('Loss - CNN Baseline', fontweight='bold')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(history_cnn['train_acc'], label='Train', color='blue')
axes[1,1].plot(history_cnn['test_acc'], label='Test', color='red')
axes[1,1].set_title('Accuracy - CNN Baseline', fontweight='bold')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('comparaison_resultats.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '='*65)
print('  RESULTATS FINAUX')
print('='*65)
print(f'  {"Modele":<25} | {"Accuracy":>10}')
print('  ' + '-'*40)
print(f'  {"ΨTA (NeuroFractal)":<25} | {best_acc_psita:>9.2f}%')
print(f'  {"CNN Baseline":<25} | {best_acc_cnn:>9.2f}%')
print(f'  {"Gain ΨTA":<25} | {"+" + f"{best_acc_psita-best_acc_cnn:.2f}%":>10}')
print('='*65)
print(f'\nGraphique sauvegarde : comparaison_resultats.png')